# Well trajectory examples

`geodetic_engine.welltrajectory` computes the path of a well from its directional survey: the measured depth (MD), inclination and azimuth at each survey station. The survey and its settings go in a `TrajectoryInput`, and `compute()` turns it into a `WellTrajectory`, with the position, TVD and dogleg severity at every station.

Every input needs four things:

| Setting | What it is |
| --- | --- |
| survey | MD, inclination and azimuth at each station |
| `wellhead` | `(x, y, z)`: the position in the CRS, and the elevation MD and TVD count from |
| `crs` | the trajectory CRS, such as `"EPSG:32631"` |
| `north_reference` | what the azimuths are measured from: grid north `"GN"` or true north `"TN"` |

The rest have defaults: `md_unit`, `angle_unit` and `z_unit`, the georeferencing `method`, `md_step` and `md_points` for extra points, and a `name`.

## Contents

1. [Reading the synthetic well](#1-reading-the-synthetic-well)
2. [Computing the trajectory](#2-computing-the-trajectory)
3. [Points between the stations](#3-points-between-the-stations)
4. [Latitude and longitude in WGS 84](#4-latitude-and-longitude-in-wgs-84)
5. [Plotting](#5-plotting)
6. [Other ways to build the input](#6-other-ways-to-build-the-input)
7. [A real well: Volve F-1](#7-a-real-well-volve-f-1)

## 1. Reading the synthetic well

The package ships a synthetic survey file: its settings in a `# key: value` header, then MD, inclination and azimuth at each station. `from_csv` reads it, and also reads files exported by other programs, with other column names and separators.

In [1]:
from importlib.resources import files

from geodetic_engine.welltrajectory import TrajectoryInput

survey_file = (
    files("geodetic_engine.welltrajectory") / "example_data" / "synthetic_well.csv"
)
well = TrajectoryInput.from_csv(survey_file)

print(well.name, well.crs, well.wellhead, well.north_reference, sep=" | ")
well.to_dataframe().head()

Synthetic-1 | EPSG:23031 | Wellhead(x=455000.0, y=6785000.0, z=32.0) | GN


,md,inclination,azimuth
0,0.0,0.00,0.00
1,30.0,0.31,210.01
2,60.0,0.30,206.73
3,90.0,0.33,201.27
4,120.0,0.46,201.99


## 2. Computing the trajectory

`compute()` gives the `WellTrajectory`. At every station it has the angles, the offsets from the wellhead (`east`, `north`, `tvd`), the position in the CRS (`x`, `y`, `z`) and the dogleg severity. `operations` records how it was computed.

In [2]:
trajectory = well.compute()

print(*trajectory.operations, sep="\n")
trajectory.to_dataframe().tail()

trajectory CRS EPSG:23031; wellhead at (455000, 6785000), elevation 32 m
grid convergence -0.733563786 deg and scale factor 0.999624807 at the wellhead
azimuths against grid north, turned onto true north by the grid convergence
minimum curvature over 122 survey stations
read offsets as ED50 / Azimuthal Equidistant Lat=61.19538608 Lon=2.16286831
converted from ED50 / Azimuthal Equidistant Lat=61.19538608 Lon=2.16286831 to ED50 / UTM zone 31N


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
117,3510.0,35.04,131.146436,131.88,1558.045077,-1186.534867,2696.629957,456542.147146,6.783794e+06,-2664.629957,0.264798,True
118,3540.0,35.24,130.616436,131.35,1571.100294,-1197.835987,2721.162445,456555.051757,6.783783e+06,-2689.162445,0.364771,True
119,3570.0,35.11,131.136436,131.87,1584.167786,-1209.145834,2745.684386,456567.968526,6.783771e+06,-2713.684386,0.326550,True
120,3600.0,35.00,130.736436,131.47,1597.184508,-1220.435878,2770.242447,456580.834802,6.783760e+06,-2738.242447,0.254720,True
121,3612.0,34.94,131.526436,132.26,1602.364789,-1224.959885,2780.075925,456585.954814,6.783755e+06,-2748.075925,1.141855,True


## 3. Points between the stations

Ask for extra points when the input is made, with `md_step` for a point every so far and `md_points` for points at given depths, or call `interpolate()` on a trajectory. The points lie on the minimum curvature arcs, and are flagged with `is_survey_station` False.

In [3]:
every_10 = TrajectoryInput.from_csv(survey_file, md_step=10).compute()
print(f"{len(trajectory)} survey stations, {len(every_10)} points with md_step=10")

trajectory.interpolate([1234.5, 2000.0]).to_dataframe()

122 survey stations, 363 points with md_step=10


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,1234.5,51.963497,127.166947,127.900511,305.826591,-202.782128,1102.153687,455303.091569,6.784793e+06,-1070.153687,0.105557,False
1,2000.0,51.979383,126.996141,127.729704,785.725581,-567.734596,1573.839002,455778.100422,6.784422e+06,-1541.839002,0.649126,False


## 4. Latitude and longitude in WGS 84

Positions are computed on the CRS's own datum, here ED50. Going to WGS 84 is a datum change, and the transformation must be named: here EPSG:1133, ED50 to WGS 84 (1).

In [4]:
wgs84 = trajectory.to_geographic(operation="EPSG:1133")

print(wgs84.operation.name)
wgs84.coordinates.to_dataframe().tail()

ED50 to WGS 84 (1)


,Lon,Lat
117,2.190072,61.184231
118,2.190315,61.184130
119,2.190557,61.184028
120,2.190799,61.183927
121,2.190896,61.183886


## 5. Plotting

Plotting needs the `plot` extra: `pip install 'geodetic-engine[plot]'`. `plot()` draws the trajectory in 3D, coloured by dogleg severity or MD, with all three axes at one scale. Drag to rotate, scroll to zoom, and hover for the values at a point. `open_in_browser()` opens the same figure full screen in a browser tab, and `plot_trajectory(a, b, ...)` draws several wells on one set of axes.

In [5]:
figure = trajectory.plot(color_by="dls")
figure

In [6]:
from geodetic_engine.welltrajectory import open_in_browser

print(open_in_browser(figure))

http://127.0.0.1:44981/326SvwcqfjlsGXzf5bJJhQ/ED50-UTM-zone-31N-1.html


The figure is a plain plotly figure: restyle it with `figure.update_layout(...)`, write it to a self-contained HTML file to keep or share with `figure.write_html(path)`, or to a still PNG, SVG or PDF with `figure.write_image(path)`.

## 6. Other ways to build the input

Each constructor takes the stations in its own form and the settings as keyword arguments, and gives a `TrajectoryInput` to `compute()` as above.

### From arrays

One array or list per quantity. Leave `azimuth` out for an inclination-only survey.

`method` sets how the offsets from the wellhead are placed in the CRS, and every constructor takes it: a member of `Method`, or its name as text. `AzimuthalEquidistant` is the default; hover over a member of `Method` for what each one does.

In [7]:
from geodetic_engine.welltrajectory import Method


well = TrajectoryInput.from_arrays(
    md=[0.0, 500.0, 1500.0, 2500.0],
    inclination=[0.0, 20.0, 60.0, 70.0],
    azimuth=[10.0, 30.0, 45.0, 50.0],
    wellhead=(500000.0, 6600000.0, 25.0),
    crs="EPSG:32631",
    north_reference="GN",
    method=Method.GRID_NORTH_LOCAL,  # or "GridNorthLocal"
)
well.to_dataframe()

,md,inclination,azimuth
0,0.0,0.0,10.0
1,500.0,20.0,30.0
2,1500.0,60.0,45.0
3,2500.0,70.0,50.0


### From rows

One row per station: a tuple `(md, inclination, azimuth)`, or a dict, which is how JSON usually holds stations.

In [8]:
stations = [
    {"md": 0, "inclination": 0, "azimuth": 10},
    {"md": 500, "inclination": 20, "azimuth": 30},
    {"md": 1500, "inclination": 60, "azimuth": 45},
]
well = TrajectoryInput.from_records(
    stations,
    wellhead=(500000.0, 6600000.0, 25.0),
    crs="EPSG:32631",
    north_reference="GN",
)
well.to_dataframe()

,md,inclination,azimuth
0,0.0,0.0,10.0
1,500.0,20.0,30.0
2,1500.0,60.0,45.0


### From a DataFrame

Name the columns to read; other columns are ignored.

In [9]:
import pandas as pd

survey = pd.DataFrame(
    {
        "MD": [0.0, 500.0, 1500.0, 2500.0],
        "INC": [0.0, 20.0, 60.0, 70.0],
        "AZI": [10.0, 30.0, 45.0, 50.0],
        "TOOL": ["gyro", "MWD", "MWD", "MWD"],
    }
)
well = TrajectoryInput.from_dataframe(
    survey,
    md_column="MD",
    inclination_column="INC",
    azimuth_column="AZI",
    wellhead=(500000.0, 6600000.0, 25.0),
    crs="EPSG:32631",
    north_reference="GN",
)
well.to_dataframe()

,md,inclination,azimuth
0,0.0,0.0,10.0
1,500.0,20.0,30.0
2,1500.0,60.0,45.0
3,2500.0,70.0,50.0


### From an OSDU request body

The body of an OSDU `convertTrajectory` request, as a dict or as JSON text. `referencePoint` is the wellhead, `trajectoryCRS` the CRS, `azimuthReference` the north reference, and `unitZ` and `unitMD` the units.

In [10]:
body = {
    "trajectoryCRS": "EPSG:32631",
    "azimuthReference": "GN",
    "unitZ": "m",
    "referencePoint": {"x": 500000.0, "y": 6600000.0, "z": 25.0},
    "inputStations": [
        {"md": 0, "inclination": 0, "azimuth": 10},
        {"md": 500, "inclination": 20, "azimuth": 30},
        {"md": 1500, "inclination": 60, "azimuth": 45},
    ],
}
well = TrajectoryInput.from_osdu_payload(body)
well.to_dataframe()

,md,inclination,azimuth
0,0.0,0.0,10.0
1,500.0,20.0,30.0
2,1500.0,60.0,45.0


## 7. A real well: Volve F-1

The survey report of well F-1 on the Volve field, a plain-text listing from Statoil's well database. `from_csv` reads it as it is: it skips the free text above the table, taking the well's name and the north reference from it, splits the columns lined up with spaces, and takes their units from the line below the names. Two settings are passed: the CRS, which the report names only in words, ED50 and UTM zone 31N, and the wellhead, where the survey starts: on the seabed, 91 m below sea level, at the report's Surface EW and NS.

Data: the Volve field data set, Equinor and the Volve licence partners, under the Equinor Open Data Licence.

In [11]:
report = (
    files("geodetic_engine.welltrajectory") / "example_data" / "volve_f1_survey.txt"
)
f1 = TrajectoryInput.from_csv(
    report, wellhead=(435046.488, 6478566.687, -91.0), crs="EPSG:23031"
).compute()

print(f1.name, f1.north_reference, f1.md_unit, sep=" | ")
f1.to_dataframe().tail()

F-1 | GN | m


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
95,3499.9,21.06,300.781928,301.73,-94.071336,876.157458,3061.134053,434966.954952,6.479444e+06,-3152.134053,0.292300,True
96,3540.5,21.02,300.611928,301.56,-106.605088,883.598569,3099.027254,434954.550369,6.479452e+06,-3190.027254,0.053921,True
97,3580.5,20.88,301.011928,301.96,-118.888524,890.924248,3136.382977,434942.394071,6.479459e+06,-3227.382977,0.150103,True
98,3620.3,20.89,301.511928,302.46,-131.016743,898.287303,3173.568060,434930.393534,6.479467e+06,-3264.568060,0.134568,True
99,3632.0,20.89,301.511928,302.46,-134.573441,900.467869,3184.498981,434926.874631,6.479469e+06,-3275.498981,0.000000,True


In [12]:
f1.plot()